# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayuj5/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import subprocess, os
from google.colab import userdata

# Clone repo
if not os.path.exists('/content/flyrank-internship-ml'):
    subprocess.run(['git', 'clone', 'https://github.com/sayuj5/flyrank-internship-ml.git'],
                   capture_output=True, text=True)
    print("Repo cloned")
else:
    print("Repo already exists")

# HF login
import huggingface_hub
token = userdata.get('HF_TOKEN')
huggingface_hub.login(token=token, add_to_git_credential=False)
print("HF login done")

Repo already exists
HF login done


In [18]:
import subprocess, os
from google.colab import userdata

# Clone repo
if not os.path.exists('/content/flyrank-internship-ml'):
    subprocess.run(['git', 'clone', 'https://github.com/sayuj5/flyrank-internship-ml.git'],
                   capture_output=True, text=True)
    print("Repo cloned")
else:
    print("Repo already exists")

# HF login
import huggingface_hub
token = userdata.get('HF_TOKEN')
huggingface_hub.login(token=token, add_to_git_credential=False)
print("HF login done")

Repo already exists
HF login done


In [19]:
import pandas as pd
from datasets import load_dataset

# Use streaming to avoid downloading 1.17GB
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    token=token,
    streaming=True
)

# Take just 50000 rows to work with
rows = []
for i, row in enumerate(dataset):
    if i >= 50000:
        break
    rows.append(row)

df = pd.DataFrame(rows)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nAvailable months: {sorted(df['month'].unique()) if 'month' in df.columns else 'no month col'}")
df.head(3)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Shape: (50000, 30)
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

Available months: no month col


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
import pandas as pd

# Filter to mid-panel month: March 2026
df['report_date'] = pd.to_datetime(df['report_date'])
df_mar = df[(df['report_date'] >= '2026-03-01') & (df['report_date'] <= '2026-03-31')].copy()

print(f"Full sample shape: {df.shape}")
print(f"March 2026 slice: {df_mar.shape}")
print(f"Date range in March slice: {df_mar['report_date'].min()} to {df_mar['report_date'].max()}")
df_mar.head(3)

Full sample shape: (50000, 30)
March 2026 slice: (0, 30)
Date range in March slice: NaT to NaT


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events


In [21]:
print("Date range in full sample:")
print(f"Min: {df['report_date'].min()}")
print(f"Max: {df['report_date'].max()}")
print(f"\nUnique months in sample:")
print(sorted(df['report_date'].dt.to_period('M').unique()))

Date range in full sample:
Min: 2025-01-27 00:00:00
Max: 2025-02-27 00:00:00

Unique months in sample:
[Period('2025-01', 'M'), Period('2025-02', 'M')]


In [22]:
from huggingface_hub import list_repo_files

files = list(list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=token
))

# Show only the fact_content_daily_performance files
perf_files = [f for f in files if 'fact_content_daily' in f]
print(f"Available files ({len(perf_files)}):")
for f in perf_files:
    print(f)

Available files (19):
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily

In [23]:
import pandas as pd
import huggingface_hub

# Download March 2026 directly (mid-panel month)
parquet_path = huggingface_hub.hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

df_mar = pd.read_parquet(parquet_path)
df_mar['report_date'] = pd.to_datetime(df_mar['report_date'])

print(f"Shape: {df_mar.shape}")
print(f"Date range: {df_mar['report_date'].min().date()} to {df_mar['report_date'].max().date()}")
print(f"Columns: {df_mar.columns.tolist()}")
df_mar.head(3)

Shape: (9841378, 30)
Date range: 2026-03-01 to 2026-03-31
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one article × one day.

Each row in fact_content_daily_performance captures the performance of a single
piece of content (content_hash_id) for a single client (client_hash_id) on a
single day (report_date). The time window for this contract is March 2026
(2026-03-01 to 2026-03-31) — a mid-panel month, safely away from the sealed
final month of June 2026.

For the Engagement Prediction lane, daily rows are aggregated to one row per
article over the full month, using gsc_clicks as the primary engagement signal.

In [24]:
# Verify grain: each (client, content, date) should be unique
grain_check = df_mar.groupby(
    ['client_hash_id', 'content_hash_id', 'report_date']
).size()

print(f"Total rows in March 2026: {len(df_mar):,}")
print(f"Unique (client, content, date) combos: {len(grain_check):,}")
print(f"Max rows per combo: {grain_check.max()}")
print("✓ Grain confirmed: one row = one article × one day"
      if grain_check.max() == 1 else "✗ Duplicates found!")
print(f"\nDate range: {df_mar['report_date'].min().date()} to {df_mar['report_date'].max().date()}")
print(f"Days covered: {df_mar['report_date'].nunique()}")
print(f"Unique articles: {df_mar['content_hash_id'].nunique():,}")
print(f"Unique clients: {df_mar['client_hash_id'].nunique():,}")

Total rows in March 2026: 9,841,378
Unique (client, content, date) combos: 9,841,378
Max rows per combo: 1
✓ Grain confirmed: one row = one article × one day

Date range: 2026-03-01 to 2026-03-31
Days covered: 31
Unique articles: 331,437
Unique clients: 55


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature columns (knowable at decision moment):
- gsc_impressions: how often the article appeared in search — logged daily by GSC
- gsc_avg_position: average search rank — available next day via GSC export
- ga4_engaged_sessions: sessions where user stayed >10s — available daily via GA4
- scroll_events: how far users scrolled — captured client-side, exported daily
- sessions_ai: traffic from AI sources — tagged at session level, available daily

Label (proxy):
- gsc_clicks: total clicks from search aggregated over the month — this is the
  observed outcome we predict. It comes from user behavior, not an editorial rule.

Context columns (not used as features):
- client_hash_id: identifier only
- content_hash_id: identifier only
- report_date: used for windowing only

Excluded:
- ga4_pageviews: generated in the same user visit as a click — leaks the label
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other:
  mostly NaN in March 2026, too sparse to be reliable features

In [25]:
feature_cols = ['gsc_impressions', 'gsc_avg_position',
                'ga4_engaged_sessions', 'scroll_events', 'sessions_ai']
label_col = 'gsc_clicks'

print("Feature availability in March 2026:")
for col in feature_cols:
    nulls = df_mar[col].isna().sum()
    pct = nulls / len(df_mar) * 100
    print(f"  {col}: mean={df_mar[col].mean():.2f}, nulls={nulls:,} ({pct:.1f}%)")

print(f"\nLabel — {label_col}:")
print(f"  mean={df_mar[label_col].mean():.2f}")
print(f"  median={df_mar[label_col].median():.0f}")
print(f"  max={df_mar[label_col].max():.0f}")

print(f"\nExcluded — ga4_pageviews (leaky):")
print(f"  correlation with gsc_clicks: {df_mar['ga4_pageviews'].corr(df_mar['gsc_clicks']):.3f}")

Feature availability in March 2026:
  gsc_impressions: mean=28.52, nulls=0 (0.0%)
  gsc_avg_position: mean=15.83, nulls=6,230,317 (63.3%)
  ga4_engaged_sessions: mean=0.00, nulls=3,018,741 (30.7%)
  scroll_events: mean=0.03, nulls=3,018,741 (30.7%)
  sessions_ai: mean=0.00, nulls=3,018,741 (30.7%)

Label — gsc_clicks:
  mean=0.08
  median=0
  max=274

Excluded — ga4_pageviews (leaky):
  correlation with gsc_clicks: 0.342


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries on March 2026:
Query 1 — Grain: confirms one row = one article × one day
Query 2 — Row count and date span: confirms slice size and coverage  
Query 3 — Availability: filters with IS TRUE, shows usable row count

In [26]:
print("=== Query 1: Grain Check ===")
grain = df_mar.groupby(
    ['client_hash_id', 'content_hash_id', 'report_date']
).size()
duplicates = (grain > 1).sum()
print(f"Total rows: {len(df_mar):,}")
print(f"Unique (client, content, date) combos: {len(grain):,}")
print(f"Duplicate combos: {duplicates}")
print("✓ One row = one article × one day" if duplicates == 0 else "✗ Duplicates found!")

=== Query 1: Grain Check ===
Total rows: 9,841,378
Unique (client, content, date) combos: 9,841,378
Duplicate combos: 0
✓ One row = one article × one day


In [27]:
print("=== Query 2: Row Count and Date Span ===")
print(f"Rows in March 2026: {len(df_mar):,}")
print(f"Date span: {df_mar['report_date'].min().date()} → {df_mar['report_date'].max().date()}")
print(f"Days covered: {df_mar['report_date'].nunique()}")
print(f"Unique articles: {df_mar['content_hash_id'].nunique():,}")
print(f"Unique clients: {df_mar['client_hash_id'].nunique():,}")

=== Query 2: Row Count and Date Span ===
Rows in March 2026: 9,841,378
Date span: 2026-03-01 → 2026-03-31
Days covered: 31
Unique articles: 331,437
Unique clients: 55


In [28]:
print("=== Query 3: Availability Check (gsc_data_available IS TRUE) ===")
total = len(df_mar)
available = df_mar[df_mar['gsc_data_available'] == True]
print(f"Total rows in March 2026: {total:,}")
print(f"Rows where gsc_data_available IS TRUE: {len(available):,}")
print(f"Availability rate: {len(available)/total*100:.1f}%")
print(f"\nOf available rows:")
print(f"  Mean clicks: {available['gsc_clicks'].mean():.2f}")
print(f"  Mean impressions: {available['gsc_impressions'].mean():.2f}")
print(f"  Rows with >0 clicks: {(available['gsc_clicks'] > 0).sum():,}")

=== Query 3: Availability Check (gsc_data_available IS TRUE) ===
Total rows in March 2026: 9,841,378
Rows where gsc_data_available IS TRUE: 3,611,061
Availability rate: 36.7%

Of available rows:
  Mean clicks: 0.23
  Mean impressions: 77.72
  Rows with >0 clicks: 417,981


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One named limitation of this slice:

fact_content_daily_performance only contains articles that already have at least
one impression — content with zero search visibility is completely absent from
this table. The engagement prediction model is therefore trained only on content
Google has already indexed and surfaced. Truly invisible content, which may need
the most editorial attention, cannot be scored at all.

Additional limits:
- Only 36.7% of rows have gsc_data_available IS TRUE — the majority of rows
  have no usable GSC signal for that day
- ga4_engaged_sessions, scroll_events and sessions_ai are NULL for 30.7% of
  rows (clients without GA4 integration)
- Only 55 unique clients in March 2026 — results may not generalize beyond
  this client cohort
- Median clicks = 0, meaning over half the article-day rows have no clicks —
  the label is heavily zero-inflated

In [29]:
print("=== Data Limits ===")
print(f"Total rows: {len(df_mar):,}")
print(f"gsc_data_available IS TRUE: {(df_mar['gsc_data_available']==True).sum():,} ({(df_mar['gsc_data_available']==True).mean()*100:.1f}%)")
print(f"ga4_data_available IS TRUE: {(df_mar['ga4_data_available']==True).sum():,} ({(df_mar['ga4_data_available']==True).mean()*100:.1f}%)")
print(f"Rows with zero clicks: {(df_mar['gsc_clicks']==0).sum():,} ({(df_mar['gsc_clicks']==0).mean()*100:.1f}%)")
print(f"Unique clients: {df_mar['client_hash_id'].nunique()} — small cohort, limited generalizability")
print(f"scroll_events nulls: {df_mar['scroll_events'].isna().sum():,} ({df_mar['scroll_events'].isna().mean()*100:.1f}%)")
print(f"\nKey limitation: only articles with >=1 impression appear in this table.")
print(f"Zero-visibility content is invisible — and may need the most attention.")

=== Data Limits ===
Total rows: 9,841,378
gsc_data_available IS TRUE: 3,611,061 (36.7%)
ga4_data_available IS TRUE: 413,966 (4.2%)
Rows with zero clicks: 9,423,397 (95.8%)
Unique clients: 55 — small cohort, limited generalizability
scroll_events nulls: 3,018,741 (30.7%)

Key limitation: only articles with >=1 impression appear in this table.
Zero-visibility content is invisible — and may need the most attention.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.